# 02. 통합 결함 탐지 모델 학습 (Colab GPU)

`01_data_prep` 이 Drive `parts/*.zip` 에 만든 데이터로 YOLO 를 학습하고, **ONNX 로 내보내 계약 형식
(`detector_<버전>.onnx` + `.json`)으로 Drive `models/` 에 저장**한다.

| 버전 | 데이터 (`PARTS`) | 클래스 | 언제 |
|---|---|---|---|
| **v1** | `["rdd2022"]` | pothole, crack (가드레일·표지판은 아직 0개) | 11/16~17 — 11/18 인터페이스 맞춤용 |
| v2 | `None` (parts 전부: RDD + AI Hub) | 4클래스 | AI Hub 데이터 준비 후 |

모델 출력은 처음부터 **4클래스**(`pothole, crack, guardrail, sign`)다. v1 은 뒤 두 클래스 데이터가 없을 뿐이라,
정현우의 가상 블랙박스는 v1 → v2 교체 때 코드를 바꿀 필요가 없다.

**미리 할 것**: **런타임 → 런타임 유형 변경 → T4 GPU**. 세션이 끊겨도 학습은 Drive 의 `last.pt` 에서 이어진다
(위에서부터 다시 실행하면 됨).

In [ ]:
# ===== 설정 =====
DRIVE_ROOT = "/content/drive/MyDrive/daboyeo_data"
REPO_URL = "https://github.com/ParkGyeom/daboyeo.git"

VERSION = "v1"
PARTS = ["rdd2022"]     # v2: None → parts 폴더의 zip 전부
BASE_MODEL = "yolo11n.pt"  # 블랙박스 CPU 속도 때문에 n급. 속도가 남으면 yolo11s.pt 비교
EPOCHS = 60
PATIENCE = 15           # val 성능이 이만큼 안 오르면 조기 종료
IMGSZ = 640
BATCH = 32              # T4 에서 n급 640 기준. 메모리 부족(OOM)이면 16
FRACTION = 1.0          # 빠른 시험은 0.1 (데이터 10%만)

RUNS = f"{DRIVE_ROOT}/runs"      # 체크포인트 (Drive 라 세션이 끊겨도 남는다)
MODELS = f"{DRIVE_ROOT}/models"  # 최종 산출물
DATASET = "/content/dataset"     # parts 를 풀 곳 (Colab 로컬 디스크, 빠름)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os, pathlib, json, shutil, subprocess, sys, time, zipfile
for d in (RUNS, MODELS):
    pathlib.Path(d).mkdir(parents=True, exist_ok=True)
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "GPU 없음 — 런타임 유형을 T4 GPU 로 바꿔야 한다"

### 코드 설치 (`ultralytics` + 레포)

In [ ]:
REPO = "/content/daboyeo"
if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)   # 항상 최신 코드
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)
!pip install -q ultralytics onnx onnxslim onnxruntime -e {REPO}/contracts -e {REPO}/ai
sys.path[:0] = [f"{REPO}/contracts", f"{REPO}/ai"]
from daboyeo_contracts import CLASS_NAMES
from daboyeo_ai.data.yolo import write_data_yaml
from daboyeo_ai.inference import OnnxDetector
print(CLASS_NAMES)

## 1. 데이터 풀기

In [ ]:
from collections import Counter
import pandas as pd

parts_dir = pathlib.Path(DRIVE_ROOT) / "parts"
zips = sorted(parts_dir.glob("*.zip"))
if PARTS is not None:
    zips = [z for z in zips if z.stem in PARTS]
    missing = set(PARTS) - {z.stem for z in zips}
    assert not missing, f"Drive parts 에 없음: {missing} — 01_data_prep 을 먼저 끝내야 한다"
print("사용할 parts:", [z.name for z in zips])

det_root = pathlib.Path(DATASET) / "det"
for z in zips:
    marker = pathlib.Path(DATASET) / f".done_{z.stem}"
    if marker.exists():
        continue
    with zipfile.ZipFile(z) as zf:   # 탐지 데이터(det/)만 푼다. 파손 크롭(cls/)은 03 노트북에서
        zf.extractall(DATASET, [m for m in zf.namelist() if m.startswith("det/")])
    marker.touch()
data_yaml = write_data_yaml(det_root)

summary = Counter()
for txt in (det_root / "labels").glob("*/*.txt"):
    split = txt.parent.name
    summary[(split, "images")] += 1
    for line in txt.read_text().splitlines():
        summary[(split, CLASS_NAMES[int(line.split()[0])])] += 1
counts = pd.Series(summary).unstack(0).fillna(0).astype(int)
display(counts)
assert counts.get("val", pd.Series(dtype=int)).get("images", 0) > 0, "val 이미지가 없다"

## 2. 학습

- 처음: `BASE_MODEL`(COCO 사전학습)에서 시작
- 다시 실행: Drive `runs/<버전>/weights/last.pt` 가 있으면 **이어서** 학습
- 증강: 좌우 반전 O, **상하 반전 X**(노면 방향), 밝기 변화 크게(야간·역광 대비)

In [ ]:
from ultralytics import YOLO
import torch

run_dir = pathlib.Path(RUNS) / VERSION
last = run_dir / "weights" / "last.pt"

def is_finished(ckpt_path):
    # 끝난 학습의 체크포인트는 epoch=-1, optimizer=None 으로 정리된다.
    # 이걸 resume 하면 Ultralytics 가 경고만 하고 '내장 COCO 샘플로 새 학습'을 시작하므로 미리 막는다.
    ck = torch.load(ckpt_path, map_location="cpu", weights_only=False)
    return ck.get("epoch", -1) == -1 or ck.get("optimizer") is None

t = time.time()
if last.exists() and is_finished(last):
    print(f"{VERSION} 학습은 이미 끝났다 — 건너뜀 (다시 학습하려면 VERSION 을 바꾸거나 {run_dir} 를 지운다)")
elif last.exists():
    print("중단된 학습을 이어서:", last)
    YOLO(str(last)).train(resume=True)
else:
    YOLO(BASE_MODEL).train(
        data=str(data_yaml), epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
        project=RUNS, name=VERSION, exist_ok=True, seed=0, workers=2,
        fliplr=0.5, flipud=0.0, hsv_v=0.5, plots=True,
    )
print(f"{(time.time() - t) / 60:.1f} 분")

## 3. 평가 (val / test, 클래스별)

In [ ]:
best_pt = run_dir / "weights" / "best.pt"
best = YOLO(str(best_pt))
rows = []
for split in ("val", "test"):
    m = best.val(data=str(data_yaml), split=split, imgsz=IMGSZ, batch=BATCH, plots=False, verbose=False)
    for i, c in enumerate(m.box.ap_class_index):
        rows.append(dict(split=split, cls=best.names[int(c)], P=m.box.p[i], R=m.box.r[i],
                         mAP50=m.box.ap50[i], mAP50_95=m.box.ap[i]))
    rows.append(dict(split=split, cls="all", P=m.box.mp, R=m.box.mr, mAP50=m.box.map50, mAP50_95=m.box.map))
metrics_df = pd.DataFrame(rows).round(3)
display(metrics_df)

## 4. ONNX 변환 + 결과 일치 확인

블랙박스는 ONNX 를 우리 `OnnxDetector`(전처리·NMS 직접 구현)로 돌린다. 그래서 **같은 ONNX 파일을
Ultralytics 로 돌린 결과와 박스·신뢰도가 같은지** 10장으로 확인한다 (계획 1.5 완료 기준).
PyTorch `best.pt` 의 `predict` 는 입력을 32 배수 직사각형으로 패딩해 신뢰도가 조금 달라서 기준으로 쓰지 않는다.

In [ ]:
import cv2
onnx_path = pathlib.Path(best.export(format="onnx", imgsz=IMGSZ, opset=17, simplify=True, dynamic=False))
onnx_det = OnnxDetector(onnx_path, threads=2, conf=0.25, iou=0.45)
ref_model = YOLO(str(onnx_path), task="detect")

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / max((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter, 1e-9)

val_imgs = sorted((det_root / "images" / "val").glob("*.jpg"))[:10]
n_ref = n_ok = 0
for p in val_imgs:
    img = cv2.imread(str(p))
    r = ref_model.predict(img, imgsz=IMGSZ, conf=0.25, iou=0.45, verbose=False)[0].boxes
    ours = onnx_det.raw(img)
    for c, s, box in zip(r.cls.int().tolist(), r.conf.tolist(), r.xyxy.tolist()):
        n_ref += 1
        n_ok += any(o[0] == c and iou(box, o[2:]) > 0.99 and abs(o[1] - s) < 0.01 for o in ours)
parity = dict(images=len(val_imgs), ultralytics_boxes=n_ref, matched=n_ok)
print(parity)
if n_ref and n_ok / n_ref < 0.95:
    print("⚠️ OnnxDetector 결과가 Ultralytics 와 다르다 — 이 출력을 공유할 것 (계획: 11/18 Go/No-go)")
elif n_ref == 0:
    print("val 10장에서 탐지가 하나도 없음 — 학습이 덜 됐거나 데이터 문제. 평가 표를 확인")

In [ ]:
# CPU 속도 참고값 (Colab CPU 기준 — 정식 측정은 노트북 PC 에서 bench_cpu 로)
speed = {}
frames = [cv2.imread(str(p)) for p in val_imgs]
for threads in (1, 2):
    d = OnnxDetector(onnx_path, threads=threads)
    d.detect(frames[0])  # 워밍업
    t = time.perf_counter()
    for f in frames * 3:
        d.detect(f)
    ms = (time.perf_counter() - t) / (len(frames) * 3) * 1000
    speed[f"{threads}thread_ms"] = round(ms, 1)
print(speed)

## 5. Drive `models/` 에 저장

In [ ]:
import datetime
out_onnx = pathlib.Path(MODELS) / f"detector_{VERSION}.onnx"
shutil.copy(onnx_path, out_onnx)
shutil.copy(best_pt, pathlib.Path(MODELS) / f"detector_{VERSION}.pt")

def m(split):
    r = metrics_df[(metrics_df.split == split)].set_index("cls")
    return {c: {k: float(r.loc[c, k]) for k in ("P", "R", "mAP50", "mAP50_95")} for c in r.index}

results_csv = run_dir / "results.csv"
card = {
    "version": VERSION,
    "file": out_onnx.name,
    "classes": list(CLASS_NAMES),            # class_id 순서 = 계약
    "input_size": IMGSZ,
    "input": "1x3xHxW float32 RGB 0~1 (letterbox, OnnxDetector 가 처리)",
    "recommended_conf": 0.25,                # 노출 기준 θ 는 오픈 이슈 #3 에서 따로 정한다
    "nms_iou": 0.45,
    "base_model": BASE_MODEL,
    "data_parts": PARTS or sorted(z.stem for z in zips),
    "train_counts": counts.to_dict(),
    "epochs_trained": len(pd.read_csv(results_csv)) if results_csv.exists() else None,
    "metrics": {"val": m("val"), "test": m("test")},
    "onnx_parity": parity,
    "colab_cpu_speed": speed,
    "created": datetime.datetime.now().isoformat(timespec="seconds"),
}
(pathlib.Path(MODELS) / f"detector_{VERSION}.json").write_text(json.dumps(card, ensure_ascii=False, indent=1))
print(json.dumps({k: card[k] for k in ("file", "classes", "epochs_trained", "onnx_parity", "colab_cpu_speed")}, ensure_ascii=False, indent=1))
!ls -la {MODELS}

## 다음

1. Drive `daboyeo_data/models/detector_<버전>.onnx` 와 `.json` 을 노트북 PC 의 `ai/models/` 에 내려받는다
   (`*.onnx` 는 git 에 올리지 않는다 — `.gitignore`).
2. 정현우: 가상 블랙박스에서 `OnnxDetector("ai/models/detector_v1.onnx", threads=1).detect(frame)`.
3. 위 평가 표와 `detector_<버전>.json` 을 공유한다.